In [0]:
catalog = "dbr_dev"
schema = "pkustra555_bronze"
dataset = "social_media_addiction_mental_wellbeing"

base_path = f"/Volumes/{catalog}/{schema}/streaming_lab"

input_path = f"{base_path}/input"
schema_path = f"{base_path}/schema"
checkpoint_path = f"{base_path}/checkpoints/schema_evolution"

target_table = f"{catalog}.{schema}.{dataset}_schema_evolution_bronze"

In [0]:
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "rescued_data")
        .option("header", True)
        .load(input_path)
)

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
)

query.awaitTermination()

In [0]:
display(spark.table(target_table))

In [0]:
spark.table(target_table).printSchema()

In [0]:
from pyspark.sql.functions import lit 

new_data_df = (
    spark.table(target_table)
    .drop("rescued_data")
    .limit(10)
    .withColumn("data_source", lit("schema_evolution_t"))
)

In [0]:
new_data_df.printSchema()

In [0]:
schema_test_path = f"{input_path}/schema_evolution_test"

new_data_df.coalesce(1).write.mode("overwrite").option("header", True).csv(schema_test_path)

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
)

query.awaitTermination()

The first run after addind a new column stopped with UnknownFieldException. Auto Loader updates the schema in schemaLocation. After restarting the query, the new column was ingested successfully

In [0]:
spark.table(target_table).filter(col("rescued_data").isNotNull()).count()

The rescued_data column was created successfully, all input records mathed the schema

In [0]:
spark.table(target_table).printSchema()

In [0]:
display(spark.table(target_table))

In [0]:
from pyspark.sql.functions import col 

spark.table(target_table).filter(col("data_source").isNotNull()).count()

In [0]:
query.lastProgress

In [0]:
for q in query.recentProgress:
    print(q)

Key observations from `query.recentProgress`:
- Batch ID: **1**
- Batch execution time: **5401 ms**
- Query planning time: **354 ms**
- Add batch time: **1384 ms**
- Commit batch time: **2158 ms**
- Input rows: **10**
- startOffset.seqNum: **1002**
- endOffset.seqNum: **1004**

In [0]:
print(query.lastProgress["batchId"])
print(query.lastProgress["durationMs"]["triggerExecution"])

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(once=True)
        .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.lastProgress

In [0]:
query.recentProgress

In [0]:
print(query.lastProgress["batchId"])
print(query.lastProgress["durationMs"]["triggerExecution"])

- `availableNow` processed all currently available data before terminating.
- `once` processed a single micro-batch and then terminated.
- In this experiment, `once` completed faster (641 ms) than `availableNow` (1043 ms), because only a small amount of data was available to process.

In [0]:
reload_table = f"{catalog}.{schema}.{dataset}_reload_test"
reload_checkpoint_path = f"{base_path}/checkpoints/reload_test"

In [0]:
reload_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", True)
        .load(input_path)
)

reload_query = (
    reload_stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", reload_checkpoint_path)
        .trigger(availableNow=True)
        .toTable(reload_table)
)

reload_query.awaitTermination()

In [0]:
count_before = spark.table(reload_table).count()
print(f"Number of records before: {count_before}")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {reload_table}")
dbutils.fs.rm(reload_checkpoint_path, True)

again the stream_df and reload_query

In [0]:
count_after = spark.table(reload_table).count()
print(f"Number of records before: {count_before}")
print(f"Number of records after: {count_after}")
print(f"Added records: {count_after - count_before}")

The reload test was performed using a separate target table and checkpoint location.

Results:
- Records before reload: **1510**
- Records after reload: **1510**
- Difference: **0**

Both the target table and checkpoint were removed before the second load. The identical row counts confirm that the data was reloaded successfully without creating duplicate records.